In [22]:
import sqlite3
import re

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> list[tuple[str, str]]:
    """
    严格保真英文分词（逐字符状态机）：
    - 拼回去与原句完全一致
    - 不合并、不丢弃、不修改任何字符
    """
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_words_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # ✅ 重建 words 表
    cursor.execute("DROP TABLE IF EXISTS words")
    cursor.execute("""
    CREATE TABLE words (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        verse_id TEXT NOT NULL,
        order_index INTEGER NOT NULL,
        word TEXT NOT NULL,
        type TEXT NOT NULL,
        entity_key TEXT,
        start_time REAL,
        end_time REAL,
        UNIQUE(verse_id, order_index)
    )
    """)

    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_words_verse_id
        ON words(verse_id)
    """)

    cursor.execute("""
        SELECT id, text_en
        FROM verse
        WHERE text_en IS NOT NULL
    """)
    verses = cursor.fetchall()

    inserted = 0

    for verse_id, text_en in verses:
        tokens = segment_english_preserve(text_en)

        # ✅ 保真校验（强烈建议保留）
        reconstructed = "".join(word for word, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse_id:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens):
            cursor.execute("""
                INSERT INTO words (verse_id, order_index, word, type)
                VALUES (?, ?, ?, ?)
            """, (str(verse_id), idx, token, token_type))
            inserted += 1

    conn.commit()
    conn.close()

    print("✅ words 表已成功生成")
    print(f"   总 token 数：{inserted}")


if __name__ == "__main__":
    build_words_table()

✅ words 表已成功生成
   总 token 数：6652


In [13]:
# 清空数据

import sqlite3

DB_PATH = "db/bible.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("DELETE FROM words;")
conn.commit()
conn.close()

print("✅ words 表数据已全部清空")

✅ words 表数据已全部清空


In [21]:
# 增量更新

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_words_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # ✅ 只取 verse 中存在，但 words 中不存在的 verse_id
    cursor.execute("""
        SELECT v.id, v.text_en
        FROM verse v
        WHERE v.text_en IS NOT NULL
          AND v.id NOT IN (
              SELECT DISTINCT w.verse_id
              FROM words w
          )
    """)

    verses = cursor.fetchall()
    print(f"🆕 需要增量拆分的 verse 数：{len(verses)}")

    inserted = 0

    for verse_id, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(word for word, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse_id:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens):
            try:
                cursor.execute("""
                    INSERT INTO words (
                        verse_id, order_index, word, type
                    ) VALUES (?, ?, ?, ?)
                """, (str(verse_id), idx, token, token_type))
                inserted += 1
            except sqlite3.IntegrityError:
                # ✅ 理论上不会进，除非并发写入
                pass

    conn.commit()
    conn.close()

    print("✅ 增量更新完成")
    print(f"   新增 token 数：{inserted}")


if __name__ == "__main__":
    build_words_table()

🆕 需要增量拆分的 verse 数：0
✅ 增量更新完成
   新增 token 数：0


In [27]:
# 从表 verse 填充表 tokens
# 增量更新（word_id = 本节中第几个 word，不含 punct/space）

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT v.id, v.book_id, v.chapter, v.verse, v.text_en
        FROM verse v
        WHERE v.text_en IS NOT NULL
          AND v.id NOT IN (
              SELECT DISTINCT SUBSTR(t.id, 1, INSTR(t.id || '.', '.', 1, 4) - 1)
              FROM tokens t
          )
    """)

    verses = cursor.fetchall()
    print(f"🆕 需要增量拆分的 verse 数：{len(verses)}")

    inserted = 0

    for verse_id, book_id, chapter, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        word_counter = 0  # ✅ 每节重置

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{verse_id}.{idx}"  # ✅ Mt.1.1.1

            # ✅ 只有 word 才分配 word_id
            if token_type == "word":
                word_counter += 1
                word_id = word_counter
            else:
                word_id = None

            cursor.execute("""
                INSERT INTO tokens (
                    id,
                    book_id,
                    chapter_id,
                    verse_id,
                    token,
                    type,
                    entity_key,
                    word_id
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                token_id,
                book_id,
                chapter,
                verse_num,
                token,
                token_type,
                None,
                word_id
            ))
            inserted += 1

    conn.commit()
    conn.close()

    print("✅ tokens 表增量更新完成")
    print(f"   新增 token 数：{inserted}")


if __name__ == "__main__":
    build_tokens_table()

🆕 需要增量拆分的 verse 数：134
✅ tokens 表增量更新完成
   新增 token 数：2975


In [31]:
# 从表 verse 填充表 tokens
# 增量更新（word_id = 本节中第几个 word，不含 punct/space）

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT v.id, v.book_id, v.chapter, v.verse, v.text_en
        FROM verse v
        WHERE v.text_en IS NOT NULL
          AND NOT EXISTS (
              SELECT 1
              FROM tokens t
              WHERE t.id LIKE v.id || '.%'
          )
    """)

    verses = cursor.fetchall()
    print(f"🆕 需要增量拆分的 verse 数：{len(verses)}")

    inserted = 0

    for verse, book_id, chapter, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        word_counter = 0

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{verse}.{idx}"

            if token_type == "word":
                word_counter += 1
                word_id = word_counter
            else:
                word_id = None

            cursor.execute("""
                INSERT INTO tokens (
                    id,
                    book_id,
                    chapter,
                    verse,
                    token,
                    type,
                    entity_key,
                    word_id
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                token_id,
                book_id,
                chapter,
                verse_num,
                token,
                token_type,
                None,
                word_id
            ))
            inserted += 1

    conn.commit()
    conn.close()

    print("✅ tokens 表增量更新完成")
    print(f"   新增 token 数：{inserted}")


if __name__ == "__main__":
    build_tokens_table()

🆕 需要增量拆分的 verse 数：134
✅ tokens 表增量更新完成
   新增 token 数：6652
